# GNN Recipe Recommendation - Link Prediction (GraphSAGE)

Recommends 5 recipes to a user based on their positive ratings (4 or 5 stars only).  
Uses a heterogeneous GNN with GraphSAGE convolutions and link prediction.  
Designed for app integration — recommendations update each time a user adds a new 4/5-star rating.

In [123]:
import pandas as pd
import numpy as np
import ast
import torch
from torch import Tensor
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv, to_hetero
import torch.nn.functional as F
from torch_geometric.loader import LinkNeighborLoader

# Fix compatibility: PyTorch 2.8 removed torch.fx._symbolic_trace.List
# which PyG 2.6.1 still references in to_hetero
import typing
import torch.fx._symbolic_trace as _st
if not hasattr(_st, 'List'):
    _st.List = typing.List
if not hasattr(_st, 'Dict'):
    _st.Dict = typing.Dict
if not hasattr(_st, 'Optional'):
    _st.Optional = typing.Optional

In [124]:
# Load recipe and user data
recipes_df = pd.read_csv('PP_recipes.csv')
users_df = pd.read_csv('PP_users.csv')

# Load interaction data (pre-split into train/val/test)
interactions_train_df = pd.read_csv('interactions_train.csv')
interactions_val_df = pd.read_csv('interactions_validation.csv')
interactions_test_df = pd.read_csv('interactions_test.csv')

# FILTER to only positive interactions (rating 4 or 5) — these mean "user likes this recipe"
interactions_train_df = interactions_train_df[interactions_train_df['rating'] >= 4].reset_index(drop=True)
interactions_val_df = interactions_val_df[interactions_val_df['rating'] >= 4].reset_index(drop=True)
interactions_test_df = interactions_test_df[interactions_test_df['rating'] >= 4].reset_index(drop=True)

# Combine all positive interactions for full graph construction
interactions_df = pd.concat([interactions_train_df, interactions_val_df, interactions_test_df], ignore_index=True)

# Drop users with fewer than 5 positive ratings — they don't have enough signal
rating_counts = interactions_df.groupby('u').size()
active_users = set(rating_counts[rating_counts >= 5].index)
before_count = len(interactions_df)
interactions_df = interactions_df[interactions_df['u'].isin(active_users)].reset_index(drop=True)
interactions_train_df = interactions_train_df[interactions_train_df['u'].isin(active_users)].reset_index(drop=True)
interactions_val_df = interactions_val_df[interactions_val_df['u'].isin(active_users)].reset_index(drop=True)
interactions_test_df = interactions_test_df[interactions_test_df['u'].isin(active_users)].reset_index(drop=True)

print(f"Recipes: {len(recipes_df)}")
print(f"Users (with >= 5 ratings): {len(active_users)}")
print(f"Positive interactions (rating >= 4): {len(interactions_df)} (dropped {before_count - len(interactions_df)} from sparse users)")
print(f"  Train: {len(interactions_train_df)}, Val: {len(interactions_val_df)}, Test: {len(interactions_test_df)}")

Recipes: 178265
Users (with >= 5 ratings): 15203
Positive interactions (rating >= 4): 635712 (dropped 26551 from sparse users)
  Train: 622423, Val: 5324, Test: 7965


In [125]:
recipes_df.head(4)

,id,i,name_tokens,ingredient_tokens,steps_tokens,techniques,calorie_level,ingredient_ids
0,424415,23,"[40480, 37229, 2911, 1019, 249, 6878, 6878, 28...","[[2911, 1019, 249, 6878], [1353], [6953], [153...","[40480, 40482, 21662, 481, 6878, 500, 246, 161...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[389, 7655, 6270, 1527, 3406]"
1,146223,96900,"[40480, 18376, 7056, 246, 1531, 2032, 40481]","[[17918], [25916], [2507, 6444], [8467, 1179],...","[40480, 40482, 729, 2525, 10906, 485, 43, 8393...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[2683, 4969, 800, 5298, 840, 2499, 6632, 7022,..."
2,312329,120056,"[40480, 21044, 16954, 8294, 556, 10837, 40481]","[[5867, 24176], [1353], [6953], [1301, 11332],...","[40480, 40482, 8240, 481, 24176, 296, 1353, 66...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...",1,"[1257, 7655, 6270, 590, 5024, 1119, 4883, 6696..."
3,74301,168258,"[40480, 10025, 31156, 40481]","[[1270, 1645, 28447], [21601], [27952, 29471, ...","[40480, 40482, 5539, 21601, 1073, 903, 2324, 4...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7940, 3609, 7060, 6265, 1170, 6654, 5003, 3561]"


In [126]:
interactions_df.head(4)

,user_id,recipe_id,date,rating,u,i
0,2312,780,2000-09-12,5.0,1674,127175
1,2312,51964,2000-09-26,5.0,1674,151793
2,2312,1232,2000-10-17,4.0,1674,15498
3,2312,4397,2000-10-17,5.0,1674,14380


In [127]:
from collections import Counter

# Build recipe features from techniques (58-dim) + calorie_level (3-dim) + ingredients (top-500 multi-hot)
recipes_df = recipes_df.sort_values('i').reset_index(drop=True)

recipe_techniques = np.array(recipes_df['techniques'].apply(ast.literal_eval).tolist())
calorie_onehot = pd.get_dummies(recipes_df['calorie_level'], prefix='calorie').values

# Build ingredient multi-hot using top 500 most common ingredients
TOP_K_INGREDIENTS = 500
parsed_ingredient_ids = recipes_df['ingredient_ids'].apply(ast.literal_eval).tolist()
ingredient_counter = Counter(ing for ids in parsed_ingredient_ids for ing in ids)
top_ingredients = [ing_id for ing_id, _ in ingredient_counter.most_common(TOP_K_INGREDIENTS)]
ingredient_to_idx = {ing_id: idx for idx, ing_id in enumerate(top_ingredients)}

ingredient_multihot = np.zeros((len(recipes_df), TOP_K_INGREDIENTS), dtype=np.float32)
for row_idx, ings in enumerate(parsed_ingredient_ids):
    for ing_id in ings:
        if ing_id in ingredient_to_idx:
            ingredient_multihot[row_idx, ingredient_to_idx[ing_id]] = 1.0

recipe_features = np.hstack([recipe_techniques, calorie_onehot, ingredient_multihot])
RECIPE_FEAT_DIM = recipe_features.shape[1]
recipe_feat = torch.from_numpy(recipe_features).to(torch.float)

print(f"Recipe features shape: {recipe_feat.shape}")
print(f"  - Techniques: {recipe_techniques.shape[1]} features")
print(f"  - Calorie levels: {calorie_onehot.shape[1]} features")
print(f"  - Ingredients (top {TOP_K_INGREDIENTS}): {ingredient_multihot.shape[1]} features")
print(f"  - Total: {RECIPE_FEAT_DIM}")

Recipe features shape: torch.Size([178265, 561])
  - Techniques: 58 features
  - Calorie levels: 3 features
  - Ingredients (top 500): 500 features
  - Total: 561


In [128]:
# Build user features from techniques (58-dim count vector)
users_df = users_df.sort_values('u').reset_index(drop=True)

user_techniques = np.array(users_df['techniques'].apply(ast.literal_eval).tolist())
user_feat = torch.from_numpy(user_techniques).to(torch.float)

print(f"User features shape: {user_feat.shape}")

User features shape: torch.Size([25076, 58])


In [129]:
# The u and i columns are already mapped to consecutive indices
num_users = users_df['u'].max() + 1
num_recipes = recipes_df['i'].max() + 1
print(f"Number of users: {num_users}")
print(f"Number of recipes: {num_recipes}")
print(f"Number of interactions: {len(interactions_df)}")

Number of users: 25076
Number of recipes: 178265
Number of interactions: 635712


In [130]:
# Build edge indices from pre-mapped user (u) and recipe (i) columns
ratings_user_id = torch.from_numpy(interactions_df['u'].values)
ratings_recipe_id = torch.from_numpy(interactions_df['i'].values)

edge_index_user_to_recipe = torch.stack([ratings_user_id, ratings_recipe_id], dim=0)
print()
print("Final edge indices pointing from users to recipes:")
print("==================================================")
print(edge_index_user_to_recipe)


Final edge indices pointing from users to recipes:
tensor([[  1674,   1674,   1674,  ...,  24942,  25006,  25047],
        [127175, 151793,  15498,  ..., 145512, 167315, 154459]])


In [131]:
edge_index_user_to_recipe.shape

torch.Size([2, 635712])

In [132]:
data = HeteroData()

# Save node indices:
data["user"].node_id = torch.arange(num_users)
data["recipe"].node_id = torch.arange(num_recipes)

# Add node features and edge indices:
data["recipe"].x = recipe_feat
data["user"].x = user_feat
data["user", "rates", "recipe"].edge_index = edge_index_user_to_recipe

# Add reverse edges from recipes to users for bidirectional message passing:
data = T.ToUndirected()(data)

In [133]:
data

HeteroData(
  user={
    node_id=[25076],
    x=[25076, 58],
  },
  recipe={
    node_id=[178265],
    x=[178265, 561],
  },
  (user, rates, recipe)={ edge_index=[2, 635712] },
  (recipe, rev_rates, user)={ edge_index=[2, 635712] }
)

# Model Training

In [134]:
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.3,
    neg_sampling_ratio=2.0,
    add_negative_train_samples=False,
    edge_types=("user", "rates", "recipe"),
    rev_edge_types=("recipe", "rev_rates", "user"),
)
train_data, val_data, test_data = transform(data)

In [135]:
# Create a mini-batch loader that generates subgraphs for GNN input
edge_label_index = train_data["user", "rates", "recipe"].edge_label_index
edge_label = train_data["user", "rates", "recipe"].edge_label
train_loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[20, 10],
    neg_sampling_ratio=2.0,
    edge_label_index=(("user", "rates", "recipe"), edge_label_index),
    edge_label=edge_label,
    batch_size=128,
    shuffle=True,
)

In [136]:
data.metadata()

(['user', 'recipe'],
 [('user', 'rates', 'recipe'), ('recipe', 'rev_rates', 'user')])

In [137]:
"""Both users and recipes have node-level features (technique vectors).
We project them into a shared hidden space via linear layers
and add learnable embeddings to improve expressiveness."""


class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x


# Dot-product classifier for edge-level predictions:
class Classifier(torch.nn.Module):
    def forward(self, x_user: Tensor, x_recipe: Tensor, edge_label_index: Tensor) -> Tensor:
        # Convert node embeddings to edge-level representations:
        edge_feat_user = x_user[edge_label_index[0]]
        edge_feat_recipe = x_recipe[edge_label_index[1]]
        # Apply dot-product to get a prediction per supervision edge:
        return (edge_feat_user * edge_feat_recipe).sum(dim=-1)


class Model(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        # Linear projections for node features:
        self.user_lin = torch.nn.Linear(58, hidden_channels)     # 58 technique features
        self.recipe_lin = torch.nn.Linear(RECIPE_FEAT_DIM, hidden_channels)  # techniques + calories + ingredients
        # Learnable embeddings to supplement features:
        self.user_emb = torch.nn.Embedding(data["user"].num_nodes, hidden_channels)
        self.recipe_emb = torch.nn.Embedding(data["recipe"].num_nodes, hidden_channels)
        # Instantiate homogeneous GNN:
        self.gnn = GNN(hidden_channels)
        # Convert GNN model into a heterogeneous variant:
        self.gnn = to_hetero(self.gnn, metadata=data.metadata())
        self.classifier = Classifier()

    def forward(self, data: HeteroData) -> Tensor:
        x_dict = {
            "user": self.user_lin(data["user"].x) + self.user_emb(data["user"].node_id),
            "recipe": self.recipe_lin(data["recipe"].x) + self.recipe_emb(data["recipe"].node_id),
        }
        # `x_dict` holds feature matrices of all node types
        # `edge_index_dict` holds all edge indices of all edge types
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        pred = self.classifier(
            x_dict["user"],
            x_dict["recipe"],
            data["user", "rates", "recipe"].edge_label_index,
        )
        return pred


model = Model(hidden_channels=64)

In [138]:
import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: '{device}'")

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1, 6):
    total_loss = total_examples = 0
    for sampled_data in tqdm.tqdm(train_loader):
        optimizer.zero_grad()
        sampled_data.to(device)
        pred = model(sampled_data)
        ground_truth = sampled_data["user", "rates", "recipe"].edge_label
        loss = F.binary_cross_entropy_with_logits(pred, ground_truth)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.numel()
        total_examples += pred.numel()
    print(f"Epoch: {epoch:03d}, Loss: {total_loss / total_examples:.4f}")

Device: 'cpu'


100%|██████████| 1192/1192 [02:48<00:00,  7.06it/s]


Epoch: 001, Loss: 2.2546


100%|██████████| 1192/1192 [04:48<00:00,  4.13it/s]


Epoch: 002, Loss: 0.3111


...

Epoch: 003, Loss: 0.2804


100%|██████████| 1192/1192 [17:42<00:00,  1.12it/s]   


Epoch: 004, Loss: 0.2645


100%|██████████| 1192/1192 [04:14<00:00,  4.68it/s]

Epoch: 005, Loss: 0.2554


# Validation & Test Evaluation

In [142]:
# Define the validation seed edges:
edge_label_index = val_data["user", "rates", "recipe"].edge_label_index
edge_label = val_data["user", "rates", "recipe"].edge_label
val_loader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[20, 10],
    edge_label_index=(("user", "rates", "recipe"), edge_label_index),
    edge_label=edge_label,
    batch_size=3 * 128,
    shuffle=False,
)
sampled_data = next(iter(val_loader))

In [143]:
from sklearn.metrics import roc_auc_score

preds = []
ground_truths = []
for sampled_data in tqdm.tqdm(val_loader):
    with torch.no_grad():
        sampled_data.to(device)
        preds.append(model(sampled_data))
        ground_truths.append(sampled_data["user", "rates", "recipe"].edge_label)

pred = torch.cat(preds, dim=0).cpu().numpy()
ground_truth = torch.cat(ground_truths, dim=0).cpu().numpy()
auc = roc_auc_score(ground_truth, pred)
print(f"Validation AUC: {auc:.4f}")

100%|██████████| 497/497 [00:52<00:00,  9.54it/s]

Validation AUC: 0.9455


In [144]:
# Test evaluation
test_edge_label_index = test_data["user", "rates", "recipe"].edge_label_index
test_edge_label = test_data["user", "rates", "recipe"].edge_label
test_loader = LinkNeighborLoader(
    data=test_data,
    num_neighbors=[20, 10],
    edge_label_index=(("user", "rates", "recipe"), test_edge_label_index),
    edge_label=test_edge_label,
    batch_size=3 * 128,
    shuffle=False,
)

preds = []
ground_truths = []
for sampled_data in tqdm.tqdm(test_loader):
    with torch.no_grad():
        sampled_data.to(device)
        preds.append(model(sampled_data))
        ground_truths.append(sampled_data["user", "rates", "recipe"].edge_label)

pred = torch.cat(preds, dim=0).cpu().numpy()
ground_truth = torch.cat(ground_truths, dim=0).cpu().numpy()
test_auc = roc_auc_score(ground_truth, pred)
print(f"Test AUC: {test_auc:.4f}")

  0%|          | 0/497 [00:00<?, ?it/s]

100%|██████████| 497/497 [00:49<00:00,  9.97it/s]

Test AUC: 0.9462


# Save Model for App Use

In [145]:
# Save the trained model and graph data needed for inference
torch.save({
    'model_state_dict': model.state_dict(),
    'data': data,
    'num_users': num_users,
    'num_recipes': num_recipes,
}, 'recipe_gnn_model.pt')
print("Model saved to recipe_gnn_model.pt")

Model saved to recipe_gnn_model.pt


# Recommendation Function

Given a user, compute scores for all recipes the user hasn't rated yet, and return the top 5.  
This function is designed to be called from an app — every time a user rates a recipe 4 or 5, call this to get updated recommendations.

In [146]:
# Build a lookup of original recipe_id from the mapped index 'i'
recipe_id_lookup = recipes_df.set_index('i')['id'].to_dict()

# Build recipe name lookup: recipe_id -> recipe name (from RAW_recipes.csv)
raw_recipes_df = pd.read_csv('RAW_recipes.csv', usecols=['id', 'name'])
recipe_name_lookup = raw_recipes_df.set_index('id')['name'].to_dict()

# Build set of recipes each user already rated (mapped indices)
user_rated_recipes = interactions_df.groupby('u')['i'].apply(set).to_dict()


def get_user_embeddings(model, data, device):
    """Compute all node embeddings from the full graph in one forward pass."""
    model.eval()
    with torch.no_grad():
        data_device = data.to(device)
        x_dict = {
            "user": model.user_lin(data_device["user"].x) + model.user_emb(data_device["user"].node_id),
            "recipe": model.recipe_lin(data_device["recipe"].x) + model.recipe_emb(data_device["recipe"].node_id),
        }
        x_dict = model.gnn(x_dict, data_device.edge_index_dict)
    return x_dict["user"].cpu(), x_dict["recipe"].cpu()


def recommend_recipes(user_idx, model, data, device, top_k=5):
    """
    Recommend top_k recipes for a given user (mapped index u).
    Excludes recipes the user already rated.
    Returns list of (recipe_id, recipe_name, score) tuples.
    """
    user_embs, recipe_embs = get_user_embeddings(model, data, device)

    # Get this user's embedding
    user_emb = user_embs[user_idx]  # shape: [hidden_channels]

    # Score all recipes via dot product
    scores = (recipe_embs * user_emb.unsqueeze(0)).sum(dim=-1)  # shape: [num_recipes]

    # Exclude already-rated recipes
    rated = user_rated_recipes.get(user_idx, set())
    for r in rated:
        scores[r] = float('-inf')

    # Get top-k recipe indices
    top_k_indices = scores.topk(top_k).indices.tolist()
    top_k_scores = scores[top_k_indices].tolist()

    # Map back to original recipe IDs and names
    results = []
    for idx, score in zip(top_k_indices, top_k_scores):
        original_id = recipe_id_lookup.get(idx, idx)
        name = recipe_name_lookup.get(original_id, "Unknown")
        results.append((original_id, name, score))

    return results

In [161]:
# Example: Get 5 recommendations for user index 0
test_user = 23
recs = recommend_recipes(test_user, model, data, device, top_k=5)

print(f"Top 5 recipe recommendations for user {test_user}:")
print("=" * 60)
for rank, (recipe_id, name, score) in enumerate(recs, 1):
    print(f"  {rank}. {name} (ID: {recipe_id}, score: {score:.4f})")

Top 5 recipe recommendations for user 23:
  1. pecan coconut sweet potaotes (ID: 468184, score: 5.4994)
  2. asparagus prosciutto rolls (ID: 50464, score: 5.2112)
  3. lime ginger chicken with homemade salsa (ID: 137197, score: 4.5861)
  4. chewy oatmeal peanut butter bars (ID: 145630, score: 4.4224)
  5. chocolate coated spoons (ID: 4020, score: 4.3948)


# Simulate: Recommendations Change When User Adds a New Rating

In the app, when a user rates a recipe 4 or 5:
1. Add the new edge to the graph
2. Re-run the model to get updated embeddings
3. Call `recommend_recipes()` again — the top 5 will change

In [162]:
def add_rating_and_recommend(user_idx, recipe_idx, data, model, device, top_k=5):
    """
    Simulate a user rating a recipe 4 or 5 in the app.
    Adds the new edge to the graph, updates the rated set,
    and returns new top-5 recommendations.
    """
    # 1. Add the new edge to the graph (both directions for message passing)
    new_edge = torch.tensor([[user_idx], [recipe_idx]])
    data["user", "rates", "recipe"].edge_index = torch.cat(
        [data["user", "rates", "recipe"].edge_index, new_edge], dim=1
    )
    rev_edge = torch.tensor([[recipe_idx], [user_idx]])
    data["recipe", "rev_rates", "user"].edge_index = torch.cat(
        [data["recipe", "rev_rates", "user"].edge_index, rev_edge], dim=1
    )

    # 2. Update the rated set so we don't recommend it again
    if user_idx not in user_rated_recipes:
        user_rated_recipes[user_idx] = set()
    user_rated_recipes[user_idx].add(recipe_idx)

    # 3. Get new recommendations with updated graph
    return recommend_recipes(user_idx, model, data, device, top_k=top_k)


# --- Demo: show how recommendations change ---
test_user = 23
print(f"=== BEFORE new rating (user {test_user}) ===")
recs_before = recommend_recipes(test_user, model, data, device, top_k=5)
for rank, (rid, name, score) in enumerate(recs_before, 1):
    print(f"  {rank}. {name} (ID: {rid}, score: {score:.4f})")

# Simulate: user rates recipe index 500 as 4 or 5 stars
new_recipe_idx = 500
original_rid = recipe_id_lookup.get(new_recipe_idx, new_recipe_idx)
new_name = recipe_name_lookup.get(original_rid, "Unknown")
print(f"\n>>> User rates \"{new_name}\" (ID: {original_rid}) with 4/5 stars <<<\n")

recs_after = add_rating_and_recommend(test_user, new_recipe_idx, data, model, device, top_k=5)
print(f"=== AFTER new rating (user {test_user}) ===")
for rank, (rid, name, score) in enumerate(recs_after, 1):
    print(f"  {rank}. {name} (ID: {rid}, score: {score:.4f})")

# Show the difference
before_ids = set(r[0] for r in recs_before)
after_ids = set(r[0] for r in recs_after)
print(f"\nRecommendations changed: {before_ids != after_ids}")

=== BEFORE new rating (user 23) ===
  1. pecan coconut sweet potaotes (ID: 468184, score: 5.4994)
  2. asparagus prosciutto rolls (ID: 50464, score: 5.2112)
  3. lime ginger chicken with homemade salsa (ID: 137197, score: 4.5861)
  4. chewy oatmeal peanut butter bars (ID: 145630, score: 4.4224)
  5. chocolate coated spoons (ID: 4020, score: 4.3948)

>>> User rates "eat this dirt cake" (ID: 319358) with 4/5 stars <<<

=== AFTER new rating (user 23) ===
  1. pecan coconut sweet potaotes (ID: 468184, score: 5.4682)
  2. asparagus prosciutto rolls (ID: 50464, score: 5.1735)
  3. lime ginger chicken with homemade salsa (ID: 137197, score: 4.5644)
  4. chewy oatmeal peanut butter bars (ID: 145630, score: 4.3975)
  5. chocolate coated spoons (ID: 4020, score: 4.3688)

Recommendations changed: False


In [164]:
# Show all recipes user 0 has rated (from the full interactions data, all splits)
all_interactions = pd.concat([
    pd.read_csv('interactions_train.csv'),
    pd.read_csv('interactions_validation.csv'),
    pd.read_csv('interactions_test.csv'),
], ignore_index=True)

user23_ratings = all_interactions[all_interactions['u'] == 23][['recipe_id', 'rating', 'date']].copy()
user23_ratings['name'] = user23_ratings['recipe_id'].map(recipe_name_lookup)
user23_ratings = user23_ratings.sort_values('rating', ascending=False).reset_index(drop=True)

print(f"User 23 — all rated recipes ({len(user23_ratings)} total):")
print("=" * 70)
for _, row in user23_ratings.iterrows():
    print(f"  [{row['rating']:.0f}★]  {row['name']}  (ID: {row['recipe_id']}, date: {row['date']})")

User 23 — all rated recipes (55 total):
  [5★]  chicken drumsticks pierre  (ID: 90739, date: 2005-04-16)
  [5★]  taco bell taco sauce  (ID: 60254, date: 2009-09-16)
  [5★]  hearty 3 grain porridge  (ID: 26423, date: 2007-09-13)
  [5★]  blt macaroni salad  (ID: 30127, date: 2007-10-16)
  [5★]  custer s old fashioned turkey rub  (ID: 265811, date: 2007-11-22)
  [5★]  ramen beef casserole  (ID: 152921, date: 2008-10-30)
  [5★]  perfectly chocolate hershey s hot cocoa  (ID: 153877, date: 2008-11-20)
  [5★]  frosted zucchini brownies  (ID: 135856, date: 2009-01-06)
  [5★]  low carb beef and broccoli stir fry  (ID: 72779, date: 2009-02-13)
  [5★]  broccoli chicken dijon  south beach diet  (ID: 120351, date: 2009-04-28)
  [5★]  tsr version of chili s grilled baby back ribs by todd wilbur  (ID: 54234, date: 2009-05-25)
  [5★]  french bread rolls to die for  (ID: 60382, date: 2009-09-16)
  [5★]  beef barley soup  (ID: 30018, date: 2009-09-16)
  [5★]  really good vegetarian meatloaf  really  (ID